# Arabic OCR Evaluation Pipeline
Evaluates a **Gemini 2.5 Flash Lite** (extraction) → **Qwen3-8B via OpenRouter** (refinement) OCR pipeline against the `bakrianoo/arabic-legal-documents-ocr-1.0` validation set.

**Ground truth source:** `val.json` → `output.full_text` field (40 usable samples out of 80 total)

**Metrics:** Character Error Rate (CER) and Word Error Rate (WER)

In [10]:
pip install -q google-genai opencv-python pillow numpy openai jiwer tqdm python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
import base64
import cv2
import io
import json
import os
import shutil
import time
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
from tqdm import tqdm
from dotenv import load_dotenv
from google import genai
from openai import OpenAI
from jiwer import cer, wer

def find_project_dir() -> Path:
    """Locate the project folder (contains .env and val.json)."""
    for path in [Path.cwd(), *Path.cwd().parents]:
        if (path / ".env").exists() or (path / "val.json").exists():
            return path
    return Path.cwd()


PROJECT_DIR = find_project_dir()
load_dotenv(PROJECT_DIR / ".env", override=True)
print(f"Using project folder: {PROJECT_DIR}")


def get_api_key(name: str) -> str:
    """Load API key from .env in the project folder."""
    key = os.getenv(name, "").strip()
    if not key:
        raise ValueError(
            f"{name} not found. Add it to {PROJECT_DIR / '.env'}:\n"
            f"{name}=your_key_here"
        )
    return key

Using project folder: c:\Users\pc\Desktop\Arabic-OCR


## 0. Flatten Images into `downloaded_images`
After extracting `downloaded_images.zip`, run this once. It copies every image from subfolders into the main folder as `0001.jpg`, `0002.jpg`, … and removes the subfolders.

In [12]:
IMAGES_DIR = str(PROJECT_DIR / "downloaded_images")
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".tif", ".tiff", ".bmp"}


def _dataset_path_key(rel_path: str) -> str:
    """Map a relative nested path to the dataset-style path used in val.json."""
    rel = rel_path.replace("\\", "/")
    if rel.startswith("pdf_images/"):
        return "/workspace/" + rel
    parts = rel.split("/")
    if len(parts) >= 2:
        return f"/workspace/pdf_images/{parts[-2]}/{parts[-1]}"
    return f"/workspace/pdf_images/{parts[-1]}"


def flatten_images_to_downloaded_images(images_dir: str) -> dict[str, str]:
    """
    Copy images from subfolders into images_dir as 0001.jpg, 0002.jpg, ...
    Saves image_path_map.json mapping dataset paths -> flat filenames.
    """
    root = Path(images_dir)
    root.mkdir(parents=True, exist_ok=True)
    map_path = root / "image_path_map.json"

    nested = sorted(
        p for p in root.rglob("*")
        if p.is_file()
        and p.suffix.lower() in IMAGE_EXTENSIONS
        and p.parent != root
    )

    if not nested:
        if map_path.exists():
            with open(map_path, encoding="utf-8") as f:
                mapping = json.load(f)
            print(f"No nested images found. Using existing map ({len(mapping)} entries).")
            return mapping
        print("No nested images found — folder already flat or empty.")
        return {}

    mapping: dict[str, str] = {}
    for i, src in enumerate(nested, start=1):
        flat_name = f"{i:04d}{src.suffix.lower()}"
        shutil.copy2(src, root / flat_name)

        rel = src.relative_to(root).as_posix()
        mapping[rel] = flat_name
        mapping[_dataset_path_key(rel)] = flat_name

    with open(map_path, "w", encoding="utf-8") as f:
        json.dump(mapping, f, ensure_ascii=False, indent=2)

    for item in root.iterdir():
        if item.is_dir():
            shutil.rmtree(item)

    print(f"Flattened {len(nested)} images into {root} (0001 … {len(nested):04d})")
    return mapping


IMAGE_PATH_MAP = flatten_images_to_downloaded_images(IMAGES_DIR)

No nested images found. Using existing map (4118 entries).


## 1. Load & Parse val.json
Extracts `(image_path, ground_truth_text)` pairs.
Only samples where `output.full_text` is populated are usable for CER/WER evaluation.

In [13]:
VAL_JSON_PATH = str(PROJECT_DIR / "val.json")

def load_eval_samples(val_json_path: str) -> list[dict]:
    """
    Parse val.json and return samples that have ground-truth full_text.

    Each sample:
        {
            'image_path': str,   # original path from dataset
            'gt_text':    str,   # ground truth Arabic text
        }
    """
    with open(val_json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    samples = []
    skipped = 0

    for item in data:
        try:
            gpt_response = json.loads(item["conversations"][1]["value"])
            full_text = gpt_response.get("output", {}).get("full_text", "")

            # Skip samples with no ground truth text
            if not full_text or not str(full_text).strip():
                skipped += 1
                continue

            samples.append({
                "image_path": item["images"][0],
                "gt_text": str(full_text).strip()
            })
        except (json.JSONDecodeError, KeyError, IndexError):
            skipped += 1

    print(f"✅ Loaded {len(samples)} eval samples with ground truth")
    print(f"⚠️  Skipped {skipped} samples (no full_text field)")
    return samples


_image_path_map_cache: dict[str, str] | None = None


def _load_image_path_map(images_dir: str) -> dict[str, str]:
    global _image_path_map_cache
    if _image_path_map_cache is None:
        map_path = os.path.join(images_dir, "image_path_map.json")
        if os.path.exists(map_path):
            with open(map_path, encoding="utf-8") as f:
                _image_path_map_cache = json.load(f)
        else:
            _image_path_map_cache = IMAGE_PATH_MAP if "IMAGE_PATH_MAP" in globals() else {}
    return _image_path_map_cache


def resolve_image_path(dataset_path: str, images_dir: str) -> str:
    """
    Map dataset image path to a flat file in downloaded_images.

    Dataset paths look like: /workspace/pdf_images/0012/page_008.jpg
    After flattening they are at: {images_dir}/0001.jpg, 0002.jpg, ...
    """
    img_map = _load_image_path_map(images_dir)
    flat_name = img_map.get(dataset_path)
    if flat_name:
        candidate = os.path.join(images_dir, flat_name)
        if os.path.exists(candidate):
            return candidate

    return None


samples = load_eval_samples(VAL_JSON_PATH)
print(f"\nAvg ground truth length: {sum(len(s['gt_text']) for s in samples) // len(samples)} chars")

✅ Loaded 40 eval samples with ground truth
⚠️  Skipped 40 samples (no full_text field)

Avg ground truth length: 1013 chars


## 2. OCR Pipeline
Gemini 2.5 Flash Lite (extraction) → Qwen3-8B via OpenRouter (refinement) OCR pipeline.

Requires `GEMINI_API_KEY` and `QWEN_API_KEY` in your `.env` file.

In [ ]:
def preprocess_image(image_path: str) -> Image.Image:
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Could not load image: {image_path}")
    gray     = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    denoised = cv2.fastNlMeansDenoising(gray)
    clahe    = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(denoised)
    binary   = cv2.adaptiveThreshold(
        enhanced, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY, 31, 11
    )
    return Image.fromarray(binary)


def image_to_data_url(image: Image.Image) -> str:
    if image.mode != "RGB":
        image = image.convert("RGB")
    buf = io.BytesIO()
    image.save(buf, format="JPEG")
    b64 = base64.standard_b64encode(buf.getvalue()).decode("ascii")
    return f"data:image/jpeg;base64,{b64}"



def extract_text_gemini(
    image: Image.Image,
    model: str = "gemini-2.5-flash-lite",
) -> str:
    """Extract all Arabic text from an image using Gemini API."""
    client = genai.Client(api_key=get_api_key("GEMINI_API_KEY"))
    prompt = """Extract all Arabic text from this image exactly as written.
Instructions:
- Preserve line breaks.
- Preserve punctuation.
- Preserve numbers exactly.
- Do NOT correct spelling mistakes.
- Do NOT translate the text.
- Return ONLY the extracted text.
"""
    response = client.models.generate_content(
        model=model,
        contents=[prompt, image],
    )
    return response.text.strip()


def refine_text_with_qwen(text: str, model: str = "qwen/qwen3-8b") -> str:
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=os.getenv("QWEN_API_KEY", "").strip(),
    )
    prompt = f"""You are an Arabic OCR correction system.
Rules:
- Correct OCR mistakes only.
- Preserve meaning, names, dates, numbers, and phone numbers.
- Preserve line breaks.
- Do not summarize or rewrite.
- Return only the corrected text.
- Do NOT add any thinking or reasoning. Output the corrected text only.

Text:
{text}
"""
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        extra_body={"thinking": False},
    )
    return response.choices[0].message.content.strip()

def run_ocr_pipeline(
    image_path: str,
    use_refinement: bool = True
) -> dict:
    """
    Run full OCR pipeline on a single image.

    Returns dict with:
        - extracted_text: raw extraction output
        - refined_text: after refinement (if enabled)
        - error:        error message if pipeline failed
    """
    result = {"extracted_text": "", "refined_text": "", "error": None}
    try:
        processed = preprocess_image(image_path)
        
        # Determine extraction model based on USE_GEMINI_LITE flag
        use_gemini_lite = globals().get("USE_GEMINI_LITE", True)
        if use_gemini_lite:
            extracted_text = extract_text_gemini(processed, model="gemini-2.5-flash-lite")
        else:
            extracted_text = extract_text_qwen(processed)
            
        result["extracted_text"] = extracted_text

        if use_refinement:
            result["refined_text"] = refine_text_with_qwen(extracted_text)
        else:
            result["refined_text"] = extracted_text

    except Exception as e:
        result["error"] = str(e)

    return result

## 3. Metrics

In [15]:
import re
import unicodedata

def normalize_arabic(text: str) -> str:
    """
    Normalize Arabic text before CER/WER computation:
    - Strip diacritics (tashkeel)
    - Normalize alef variants → ا
    - Normalize teh marbuta → ه
    - Collapse extra whitespace
    """
    # Remove diacritics (harakat)
    diacritics = re.compile(r'[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06DC\u06DF-\u06E4\u06E7\u06E8\u06EA-\u06ED]')
    text = diacritics.sub('', text)

    # Normalize alef variants
    text = re.sub('[\u0622\u0623\u0625\u0671]', '\u0627', text)  # → ا
    text = text.replace('\u0629', '\u0647')  # ة → ه
    text = text.replace('\u0649', '\u064A')  # ى → ي

    # Collapse whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def compute_metrics(hypothesis: str, reference: str) -> dict:
    """Compute CER and WER between hypothesis and reference."""
    hyp = normalize_arabic(hypothesis)
    ref = normalize_arabic(reference)

    if not ref.strip():
        return {"cer": None, "wer": None}

    return {
        "cer": round(cer(ref, hyp), 4),
        "wer": round(wer(ref, hyp), 4)
    }

## 4. Run Evaluation

Set `USE_REFINEMENT = False` to evaluate extraction only (Gemini 2.5 Flash Lite).
Set `USE_REFINEMENT = True` to run the full pipeline with Qwen3-8B refinement via OpenRouter.

Set `MAX_SAMPLES` to limit the run (e.g. `10` for a quick test, `None` for all 40).

In [16]:
USE_GEMINI_LITE = True   # Use Gemini 2.5 Flash Lite for extraction and refinement
USE_REFINEMENT  = True   # Run refinement on extracted text
MAX_SAMPLES     = 15     # Reduce evaluation set to 15 samples (evenly spaced)
SLEEP_BETWEEN   = 0      # Inter-iteration sleep (we use per-call sleeps instead)


def select_evenly_spaced_samples(samples: list[dict], k: int) -> list[dict]:
    """Select `k` samples evenly spaced across `samples`.
    If k >= len(samples) the original list is returned.
    """
    if k is None or k >= len(samples):
        return samples
    indices = np.linspace(0, len(samples) - 1, num=k).round().astype(int)
    # Ensure unique, sorted indices
    indices = sorted(set(indices.tolist()))
    return [samples[i] for i in indices]


eval_samples = select_evenly_spaced_samples(samples, MAX_SAMPLES) if MAX_SAMPLES else samples
results = []

print(f"Running eval on {len(eval_samples)} samples (refinement={'ON' if USE_REFINEMENT else 'OFF'})\n")

for i, sample in enumerate(tqdm(eval_samples, desc="OCR Eval")):
    # Progress indicator to show pipeline is active during sleeps
    print(f"Processing image {i+1}/{len(eval_samples)}...")
    local_path = resolve_image_path(sample["image_path"], IMAGES_DIR)

    row = {
        "index":      i,
        "image_path": sample["image_path"],
        "gt_text":    sample["gt_text"],
        "extracted_text": "",
        "refined_text": "",
        "cer_extracted":  None,
        "wer_extracted":  None,
        "cer_refined": None,
        "wer_refined": None,
        "error":       None
    }

    if local_path is None:
        row["error"] = f"Image not found: {sample['image_path']}"
        results.append(row)
        continue

    ocr_result = run_ocr_pipeline(local_path, use_refinement=USE_REFINEMENT)

    if ocr_result["error"]:
        row["error"] = ocr_result["error"]
        results.append(row)
        continue

    row["extracted_text"] = ocr_result["extracted_text"]
    row["refined_text"] = ocr_result["refined_text"]

    # Metrics: extracted text vs ground truth
    m_extracted = compute_metrics(ocr_result["extracted_text"], sample["gt_text"])
    row["cer_extracted"] = m_extracted["cer"]
    row["wer_extracted"] = m_extracted["wer"]

    # Metrics: refined vs ground truth
    if USE_REFINEMENT:
        m_refined = compute_metrics(ocr_result["refined_text"], sample["gt_text"])
        row["cer_refined"] = m_refined["cer"]
        row["wer_refined"] = m_refined["wer"]

    results.append(row)
    time.sleep(SLEEP_BETWEEN)

print("\n✅ Eval complete")

Running eval on 15 samples (refinement=ON)



OCR Eval:   0%|          | 0/15 [00:00<?, ?it/s]

Processing image 1/15...


OCR Eval:   7%|▋         | 1/15 [00:01<00:19,  1.39s/it]

Processing image 2/15...


...

Processing image 14/15...


OCR Eval:  93%|█████████▎| 14/15 [00:49<00:01,  1.30s/it]

Processing image 15/15...


OCR Eval: 100%|██████████| 15/15 [00:49<00:00,  3.32s/it]


✅ Eval complete


## 5. Aggregate Results

The table below summarizes all evaluation metrics tracked for extracted and refined output.

| Metric | Description |
| --- | --- |
| `cer_extracted` | CER between Gemini 2.5 Flash Lite extraction and ground truth |
| `wer_extracted` | WER between Gemini 2.5 Flash Lite extraction and ground truth |
| `cer_refined` | CER between Qwen3-8B refined output and ground truth |
| `wer_refined` | WER between Qwen3-8B refined output and ground truth |

In [17]:
df = pd.DataFrame(results)

# Separate successful vs failed
df_ok   = df[df["error"].isna()]
df_fail = df[df["error"].notna()]

print(f"{'='*55}")
print(f"  ARABIC OCR EVALUATION SUMMARY")
print(f"{'='*55}")
print(f"  Total samples:   {len(df)}")
print(f"  Successful:      {len(df_ok)}")
print(f"  Failed/skipped:  {len(df_fail)}")
print(f"{'='*55}")

if len(df_ok) > 0:
    cer_e = df_ok["cer_extracted"].dropna().mean()
    wer_e = df_ok["wer_extracted"].dropna().mean()
    print(f"  Extracted-only →  CER: {cer_e:.4f}  |  WER: {wer_e:.4f}")

    # Dynamic description based on active model flag
    use_gemini_lite = globals().get("USE_GEMINI_LITE", True)
    extraction_desc = "Gemini 2.5 Flash Lite raw extraction vs ground truth" if use_gemini_lite else "qwen2.5vl:7b-q4_K_M (Ollama) raw extraction vs ground truth"
    refinement_desc = "Gemini 2.5 Flash Lite refined output vs ground truth" if use_gemini_lite else "Gemini 2.5 Flash refined output vs ground truth"

    summary_rows = [
        {
            "Metric": "CER Extracted",
            "Description": extraction_desc,
            "Value": cer_e,
        },
        {
            "Metric": "WER Extracted",
            "Description": extraction_desc,
            "Value": wer_e,
        },
    ]

    if USE_REFINEMENT:
        cer_r = df_ok["cer_refined"].dropna().mean()
        wer_r = df_ok["wer_refined"].dropna().mean()
        cer_delta = cer_e - cer_r
        wer_delta = wer_e - wer_r
        print(f"  Extracted+Flash →  CER: {cer_r:.4f}  |  WER: {wer_r:.4f}")
        print(f"  Flash impact  →  ΔCER: {cer_delta:+.4f}  |  ΔWER: {wer_delta:+.4f}")
        print(f"  (positive = refinement improved score)")

        summary_rows.extend([
            {
                "Metric": "CER Refined",
                "Description": refinement_desc,
                "Value": cer_r,
            },
            {
                "Metric": "WER Refined",
                "Description": refinement_desc,
                "Value": wer_r,
            },
            {
                "Metric": "ΔCER",
                "Description": "CER improvement from refinement (positive = better)",
                "Value": cer_delta,
            },
            {
                "Metric": "ΔWER",
                "Description": "WER improvement from refinement (positive = better)",
                "Value": wer_delta,
            },
        ])

    summary_df = pd.DataFrame(summary_rows)
    print("\nNumeric results summary:")
    display(summary_df)

print(f"{'='*55}")

if len(df_fail) > 0:
    print(f"\nFailed samples:")
    for _, row in df_fail.iterrows():
        print(f"  [{row['index']}] {row['error']}")

  ARABIC OCR EVALUATION SUMMARY
  Total samples:   15
  Successful:      0
  Failed/skipped:  15

Failed samples:
  [0] Error code: 400 - {'error': {'message': 'qwen3:8b is not a valid model ID', 'code': 400}, 'user_id': 'user_3EMrZJxTn6heskqx92SqV0P7tHy'}
  [1] 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite\nPlease retry in 50.407613666s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc

## 6. Per-Sample Breakdown

In [18]:
# Sort by CER (worst first) to identify hardest samples
col = "cer_refined" if USE_REFINEMENT else "cer_extracted"
df_ok_sorted = df_ok.sort_values(col, ascending=False)

display(
    df_ok_sorted[[
        "index", "image_path",
        "cer_extracted", "wer_extracted",
        "cer_refined", "wer_refined"
    ]].reset_index(drop=True)
)

,index,image_path,cer_extracted,wer_extracted,cer_refined,wer_refined


## 7. Inspect a Specific Sample

In [19]:
INSPECT_INDEX = 0  # Change to inspect different samples

row = df_ok_sorted.iloc[INSPECT_INDEX]
print(f"Image: {row['image_path']}")
print(f"CER (Extracted): {row['cer_extracted']}  |  WER (Extracted): {row['wer_extracted']}")
if USE_REFINEMENT:
    print(f"CER (Refined): {row['cer_refined']}  |  WER (Refined): {row['wer_refined']}")
print()
print("--- GROUND TRUTH ---")
print(row["gt_text"])
print()
print("--- EXTRACTED OUTPUT ---")
print(row["extracted_text"])
if USE_REFINEMENT:
    print()
    print("--- REFINED OUTPUT ---")
    print(row["refined_text"])

IndexError: single positional indexer is out-of-bounds

## 8. Save Results

In [ ]:
OUTPUT_PATH = str(PROJECT_DIR / "ocr_eval_results.csv")
df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print(f"Results saved to {OUTPUT_PATH}")

json_path = PROJECT_DIR / "ocr_eval_results.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print(f"Full results saved to {json_path}")